# 🎯 Conteo de Personas por Frame con YOLOv8
Este notebook detecta y cuenta personas en cada frame de un video usando YOLOv8.

## 1. Instalación de dependencias

In [1]:
!pip install ultralytics opencv-python-headless tqdm

## 2. Importar librerías

In [3]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm
from IPython.display import display, Image
import os

print(' Librerías importadas correctamente')

 Librerías importadas correctamente


## 3. Configuración

In [9]:
# ─── CAMBIA ESTOS VALORES ───────────────────────────────────────────
VIDEO_PATH   = 'data/videos/videoTM_02.mkv'   # Ruta a tu video
OUTPUT_VIDEO = 'outputs/videos/output.mp4'     # Video de salida con anotaciones
OUTPUT_CSV   = 'conteo.csv'     # CSV con conteo por frame

MODEL_SIZE   = 'yolov8l.pt'     # n=nano(rápido) | s=small | m=medium | l=large
CONFIDENCE   = 0.7              # Confianza mínima (0.0 – 1.0)
SKIP_FRAMES  = 1             # Procesar 1 de cada N frames (1 = todos)
# ────────────────────────────────────────────────────────────────────

assert os.path.exists(VIDEO_PATH), f'No se encontró el video: {VIDEO_PATH}'
print(f'Video encontrado: {VIDEO_PATH}')

Video encontrado: data/videos/videoTM_02.mkv


## 4. Cargar modelo YOLOv8

In [5]:
model = YOLO(MODEL_SIZE)   # Se descarga automáticamente la primera vez
print(f' Modelo {MODEL_SIZE} cargado')

# Clase 0 en COCO = 'person'
PERSON_CLASS_ID = 0

 Modelo yolov8l.pt cargado


## 5. Procesar el video y contar personas

In [10]:
cap    = cv2.VideoCapture(VIDEO_PATH)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f' Resolución : {width}x{height}')
print(f' FPS        : {fps:.1f}')
print(f'Frames tot.: {total}')

# Escritor de video de salida
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

records     = []   # [{frame, timestamp, count}]
frame_index = 0

with tqdm(total=total, desc='Procesando') as pbar:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_index % SKIP_FRAMES == 0:
            results = model(
                frame,
                classes=[PERSON_CLASS_ID],
                conf=CONFIDENCE,
                verbose=False
            )[0]

            boxes        = results.boxes
            person_count = len(boxes)
            timestamp    = frame_index / fps

            records.append({
                'frame'    : frame_index,
                'timestamp': round(timestamp, 3),
                'personas' : person_count
            })

            # ── Dibujar bounding boxes ──────────────────────────────
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf_val = float(box.conf[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    frame, f'{conf_val:.2f}',
                    (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    (0, 255, 0), 2
                )

            # ── Contador en esquina superior izquierda ───────────────
            label = f'Personas: {person_count}'
            cv2.rectangle(frame, (10, 10), (220, 50), (0, 0, 0), -1)
            cv2.putText(
                frame, label,
                (15, 38), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (255, 255, 255), 2
            )

        out.write(frame)
        frame_index += 1
        pbar.update(1)

cap.release()
out.release()
print(f'\n Video guardado en: {OUTPUT_VIDEO}')

 Resolución : 640x480
 FPS        : 30.0
Frames tot.: 26999


Procesando: 100%|██████████| 26999/26999 [1:34:42<00:00,  4.75it/s]


 Video guardado en: outputs/videos/output.mp4


## 6. Guardar resultados en CSV

In [ ]:
df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)

print(f' CSV guardado en: {OUTPUT_CSV}')
print('\n📊 Estadísticas:')
print(f'  Máximo  de personas en un frame : {df.personas.max()}')
print(f'  Promedio de personas por frame   : {df.personas.mean():.2f}')
print(f'  Frames sin personas              : {(df.personas == 0).sum()}')
df.tail()

## 7. Gráfica de conteo a lo largo del video

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.fill_between(df.timestamp, df.personas, alpha=0.25, color='steelblue')
ax.plot(df.timestamp, df.personas, color='steelblue', linewidth=1.5, label='Personas detectadas')
ax.axhline(df.personas.mean(), color='tomato', linestyle='--', linewidth=1.2, label=f'Promedio ({df.personas.mean():.1f})')

ax.set_xlabel('Tiempo (segundos)', fontsize=12)
ax.set_ylabel('Número de personas', fontsize=12)
ax.set_title('Conteo de personas por frame', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('conteo_plot.png', dpi=150)
plt.show()
print('Gráfica guardada como conteo_plot.png')

## 8. Vista previa del frame con más personas

In [ ]:
best_frame_idx = int(df.loc[df.personas.idxmax(), 'frame'])

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, best_frame_idx)
ret, frame = cap.read()
cap.release()

if ret:
    results = model(frame, classes=[PERSON_CLASS_ID], conf=CONFIDENCE, verbose=False)[0]
    annotated = results.plot()
    preview_path = 'frame_max_personas.jpg'
    cv2.imwrite(preview_path, annotated)

    from IPython.display import Image as IPImage
    display(IPImage(preview_path, width=700))
    print(f'Frame {best_frame_idx}: {df.personas.max()} personas detectadas')